# 03 - VAE on EMG (lift-level)

Same architecture as notebook 02 but on EMG envelopes (N, 400, 8).

- **Input:** `VAE/Tensors/lifts_emg.npz` from notebook 01
- **Split:** 80/20 participant-grouped, sex-stratified (same seed as IMU)
- **Output:** `VAE/Results/vae_emg/`

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, confusion_matrix
)

ROOT = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project")
TENSOR_PATH = ROOT / "VAE" / "Tensors" / "lifts_emg.npz"
OUT_DIR = ROOT / "VAE" / "Results" / "vae_emg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

LATENT_DIM = 16
BETA = 1.0
LAMBDA_CE = 1.0
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-5
LOAD_BINS = np.array([2.3, 4.5, 6.8, 9.1, 11.3])

Device: cpu


In [2]:
data = np.load(TENSOR_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)
participant = data["participant"]
sex = data["sex"]
box_mass = data["box_mass"].astype(np.float32)
channel_names = data["channel_names"]

def mass_to_class(m):
    return int(np.argmin(np.abs(LOAD_BINS - m)))
y = np.array([mass_to_class(m) for m in box_mass], dtype=np.int64)

print("X shape:", X.shape, "subjects:", len(np.unique(participant)))
print("Class dist:", dict(zip(*np.unique(y, return_counts=True))))

X shape: (4750, 400, 8) subjects: 40
Class dist: {0: 947, 1: 960, 2: 947, 3: 948, 4: 948}


In [3]:
# ---- split (same scheme as IMU) ----
subjects = np.unique(participant)
subj_sex = {p: sex[participant == p][0] for p in subjects}
rng = np.random.default_rng(SEED)
test_subjects = []
for s_label in np.unique(list(subj_sex.values())):
    subs_s = np.array([p for p, s in subj_sex.items() if s == s_label])
    rng.shuffle(subs_s)
    n_test = max(1, int(round(0.2 * len(subs_s))))
    test_subjects.extend(subs_s[:n_test].tolist())
test_subjects = set(test_subjects)
train_subjects = set(subjects) - test_subjects

train_mask = np.array([p in train_subjects for p in participant])
test_mask = ~train_mask
print(f"Train subjects: {len(train_subjects)} lifts: {train_mask.sum()}  |  Test subjects: {len(test_subjects)} lifts: {test_mask.sum()}")

Train subjects: 32 lifts: 3791  |  Test subjects: 8 lifts: 959


In [4]:
X_train_raw = X[train_mask]
ch_mean = X_train_raw.reshape(-1, X.shape[-1]).mean(axis=0)
ch_std = X_train_raw.reshape(-1, X.shape[-1]).std(axis=0) + 1e-6
Xn = (X - ch_mean) / ch_std

X_train = torch.from_numpy(Xn[train_mask]).transpose(1, 2)
X_test = torch.from_numpy(Xn[test_mask]).transpose(1, 2)
y_train = torch.from_numpy(y[train_mask])
y_test = torch.from_numpy(y[test_mask])
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: torch.Size([3791, 8, 400]) X_test: torch.Size([959, 8, 400])


In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(in_c, out_c, k, s, p), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class DeconvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p, op):
        super().__init__()
        self.net = nn.Sequential(nn.ConvTranspose1d(in_c, out_c, k, s, p, output_padding=op), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class VAE(nn.Module):
    def __init__(self, n_channels, seq_len=400, latent_dim=16, n_classes=5):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBlock(n_channels, 32, 7, 2, 3),
            ConvBlock(32, 64, 5, 2, 2),
            ConvBlock(64, 128, 3, 2, 1),
        )
        self.enc_out_len = seq_len // 8
        self.flat_dim = 128 * self.enc_out_len
        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.flat_dim)
        self.dec = nn.Sequential(
            DeconvBlock(128, 64, 3, 2, 1, 1),
            DeconvBlock(64, 32, 5, 2, 2, 1),
            nn.ConvTranspose1d(32, n_channels, 7, 2, 3, output_padding=1),
        )
        self.cls = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(inplace=True), nn.Linear(64, n_classes))

    def encode(self, x):
        h = self.enc(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparam(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    def decode(self, z):
        return self.dec(self.fc_dec(z).view(-1, 128, self.enc_out_len))
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        return self.decode(z), mu, logvar, z, self.cls(z)

model = VAE(n_channels=X.shape[-1], seq_len=X.shape[1], latent_dim=LATENT_DIM, n_classes=len(LOAD_BINS)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 389,229


In [6]:
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
history = []

def vae_losses(x, x_rec, mu, logvar, logits, y):
    recon = F.mse_loss(x_rec, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    ce = F.cross_entropy(logits, y)
    return recon, kl, ce

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr = {"recon": 0.0, "kl": 0.0, "ce": 0.0, "n": 0, "correct": 0}
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        x_rec, mu, logvar, z, logits = model(xb)
        recon, kl, ce = vae_losses(xb, x_rec, mu, logvar, logits, yb)
        loss = recon + BETA * kl + LAMBDA_CE * ce
        loss.backward(); opt.step()
        bs = xb.size(0)
        tr["recon"] += recon.item()*bs; tr["kl"] += kl.item()*bs; tr["ce"] += ce.item()*bs
        tr["n"] += bs; tr["correct"] += (logits.argmax(1)==yb).sum().item()

    model.eval()
    te = {"recon": 0.0, "kl": 0.0, "ce": 0.0, "n": 0, "correct": 0}
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            x_rec, mu, logvar, z, logits = model(xb)
            recon, kl, ce = vae_losses(xb, x_rec, mu, logvar, logits, yb)
            bs = xb.size(0)
            te["recon"] += recon.item()*bs; te["kl"] += kl.item()*bs; te["ce"] += ce.item()*bs
            te["n"] += bs; te["correct"] += (logits.argmax(1)==yb).sum().item()

    row = {"epoch": epoch,
           "train_recon": tr["recon"]/tr["n"], "train_kl": tr["kl"]/tr["n"], "train_ce": tr["ce"]/tr["n"], "train_acc": tr["correct"]/tr["n"],
           "test_recon":  te["recon"]/te["n"], "test_kl":  te["kl"]/te["n"], "test_ce":  te["ce"]/te["n"], "test_acc":  te["correct"]/te["n"]}
    history.append(row)
    if epoch % 5 == 0 or epoch == 1:
        print(f"ep{epoch:3d} | tr_rec {row['train_recon']:.3f} kl {row['train_kl']:.3f} ce {row['train_ce']:.3f} acc {row['train_acc']:.3f} "
              f"| te_rec {row['test_recon']:.3f} kl {row['test_kl']:.3f} ce {row['test_ce']:.3f} acc {row['test_acc']:.3f}")

pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

ep  1 | tr_rec 1.016 kl 0.224 ce 1.556 acc 0.283 | te_rec 0.758 kl 0.153 ce 1.529 acc 0.299
ep  5 | tr_rec 0.870 kl 0.139 ce 1.343 acc 0.387 | te_rec 0.708 kl 0.202 ce 1.503 acc 0.306
ep 10 | tr_rec 0.825 kl 0.157 ce 1.201 acc 0.454 | te_rec 0.720 kl 0.183 ce 1.584 acc 0.332
ep 15 | tr_rec 0.817 kl 0.207 ce 1.000 acc 0.569 | te_rec 0.725 kl 0.177 ce 1.781 acc 0.312
ep 20 | tr_rec 0.817 kl 0.258 ce 0.756 acc 0.684 | te_rec 0.740 kl 0.215 ce 2.075 acc 0.334
ep 25 | tr_rec 0.812 kl 0.301 ce 0.528 acc 0.795 | te_rec 0.721 kl 0.263 ce 2.722 acc 0.334
ep 30 | tr_rec 0.810 kl 0.328 ce 0.302 acc 0.894 | te_rec 0.720 kl 0.280 ce 3.137 acc 0.317
ep 35 | tr_rec 0.802 kl 0.335 ce 0.232 acc 0.919 | te_rec 0.718 kl 0.279 ce 3.489 acc 0.313
ep 40 | tr_rec 0.797 kl 0.323 ce 0.161 acc 0.943 | te_rec 0.712 kl 0.286 ce 3.633 acc 0.323
ep 45 | tr_rec 0.799 kl 0.329 ce 0.163 acc 0.943 | te_rec 0.712 kl 0.252 ce 3.993 acc 0.286
ep 50 | tr_rec 0.792 kl 0.324 ce 0.112 acc 0.959 | te_rec 0.712 kl 0.262 ce 4.39

In [7]:
# ---- eval + fairness + save ----
model.eval()
all_mu, all_logits, all_y = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        mu, logvar = model.encode(xb)
        logits = model.cls(mu)
        all_mu.append(mu.cpu().numpy()); all_logits.append(logits.cpu().numpy()); all_y.append(yb.numpy())
mu_test = np.concatenate(all_mu)
logits_test = np.concatenate(all_logits)
y_true = np.concatenate(all_y)
y_pred = logits_test.argmax(1)
mass_true = LOAD_BINS[y_true]; mass_pred = LOAD_BINS[y_pred]

acc = accuracy_score(y_true, y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)
f1m = f1_score(y_true, y_pred, average="macro")
mae = mean_absolute_error(mass_true, mass_pred)
rmse = np.sqrt(mean_squared_error(mass_true, mass_pred))
print(f"Test  acc={acc:.3f}  bal_acc={bal_acc:.3f}  macro_F1={f1m:.3f}  MAE={mae:.3f} kg  RMSE={rmse:.3f} kg")

test_sex = sex[test_mask]
signed_err = mass_pred - mass_true
abs_err = np.abs(signed_err)
fair_rows = []
for s_label in np.unique(test_sex):
    m = test_sex == s_label
    fair_rows.append({"sex": s_label, "n": int(m.sum()),
        "acc": accuracy_score(y_true[m], y_pred[m]),
        "bal_acc": balanced_accuracy_score(y_true[m], y_pred[m]),
        "mae_kg": float(abs_err[m].mean()),
        "mean_signed_err_kg": float(signed_err[m].mean()),
        "over_rate": float((signed_err[m]>0).mean()),
        "under_rate": float((signed_err[m]<0).mean())})
fair_df = pd.DataFrame(fair_rows)
print(fair_df.to_string(index=False))

torch.save({"model_state": model.state_dict(), "ch_mean": ch_mean, "ch_std": ch_std,
            "latent_dim": LATENT_DIM, "seq_len": X.shape[1], "n_channels": X.shape[-1],
            "channel_names": list(map(str, channel_names))}, OUT_DIR / "vae_emg.pt")
np.savez(OUT_DIR / "test_outputs.npz", mu=mu_test, logits=logits_test,
         y_true=y_true, y_pred=y_pred, mass_true=mass_true, mass_pred=mass_pred,
         participant=participant[test_mask], sex=test_sex)
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump({"acc": acc, "bal_acc": bal_acc, "macro_f1": f1m,
               "mae_kg": float(mae), "rmse_kg": float(rmse),
               "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
               "fairness": fair_rows}, f, indent=2)
fair_df.to_csv(OUT_DIR / "fairness.csv", index=False)
print("Saved to:", OUT_DIR)

Test  acc=0.327  bal_acc=0.327  macro_F1=0.324  MAE=2.547 kg  RMSE=3.456 kg
   sex   n      acc  bal_acc  mae_kg  mean_signed_err_kg  over_rate  under_rate
Female 479 0.352818 0.352522 2.44739            0.061169   0.336117    0.311065
  Male 480 0.302083 0.302083 2.64625           -0.462083   0.300000    0.397917
Saved to: C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\VAE\Results\vae_emg
